# tqdm-postfix-metrics — worked example 3: Nested tqdm bars for epochs and batches

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tqdm-postfix-metrics`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Training loops often have two levels of iteration: an outer epoch loop and an inner batch loop. `tqdm` supports nesting via `tqdm(..., leave=True)` for the outer bar and `tqdm(..., leave=False)` for the inner bar. The inner bar is overwritten each epoch; the outer bar accumulates. Each bar gets its own `set_postfix` call carrying the metrics relevant to that level.

## Worked solution

**Step 1 – Outer epoch bar.** We use `epoch_bar = tqdm(range(n_epochs), desc='Epochs', leave=True)` so it persists after all epochs complete.

**Step 2 – Inner batch bar.** For each epoch we create a fresh `batch_bar = tqdm(enumerate(batches), desc='Batches', total=len(batches), leave=False)`. `leave=False` means it clears itself after each epoch, keeping the terminal tidy.

**Step 3 – Per-batch postfix.** We call `batch_bar.set_postfix(batch_loss=f'{bl:.3f}')` inside the batch loop.

**Step 4 – Per-epoch postfix.** After the inner loop finishes we compute `epoch_avg = sum(batch_losses) / len(batch_losses)` and call `epoch_bar.set_postfix(epoch_avg=f'{epoch_avg:.3f}')` so the outer bar always shows the latest epoch-level metric.

**Step 5 – Return.** A list of per-epoch average losses lets the test verify the epoch bar was updated correctly.

In [ ]:
from tqdm import tqdm
import torch as t

def worked3_nested_tqdm(n_epochs=3, n_batches=4):
    """
    Nested tqdm: outer epoch bar + inner batch bar.
    Returns list of per-epoch average batch losses.
    """
    t.manual_seed(42)
    epoch_avgs = []
    epoch_bar = tqdm(range(n_epochs), desc='Epochs', leave=True)
    for epoch in epoch_bar:
        # Simulate batch losses
        t.manual_seed(epoch)
        batches = [float(v) for v in t.rand(n_batches) + 0.5]
        batch_losses = []
        batch_bar = tqdm(enumerate(batches), desc='Batches',
                         total=len(batches), leave=False)
        for batch_idx, bl in batch_bar:
            batch_losses.append(bl)
            batch_bar.set_postfix(batch_loss=f'{bl:.3f}')
        epoch_avg = sum(batch_losses) / len(batch_losses)
        epoch_avgs.append(epoch_avg)
        epoch_bar.set_postfix(epoch_avg=f'{epoch_avg:.3f}')
    return epoch_avgs

avgs = worked3_nested_tqdm()
print('epoch averages:', [f'{v:.3f}' for v in avgs])